# Part III-c sizecheck: Llama-3.1-8B on Colab

Before registration (`docs/PREREG_weights_codec_allocation.md`), the prereg needs the 8B model's measured GPU peak.
NRP's 48 GB cards (A40, RTX A6000) were all busy on 2026-09-29, so this runs the **same pinned code** on a Colab GPU.

- `memprobe`: the harness's own per-group code on the 8B's shapes with random weights (peaks depend on shapes, not values).
- `refcheck`: the fp16 harness against the checkpoint's bf16 reference (informational) and against fp32 (the prereg's fixed rule) on the 48 scored windows.

**Runtime: GPU, A100 (80 GB preferred; 40 GB may be too small).** Everything is pinned: harness commit `37ac5cb19691`,
model `unsloth/Meta-Llama-3.1-8B` at `e9a141a2091e`, transformers 4.56.1, torch 2.8.0. Each input is checked against the
prereg's recorded hashes; any mismatch stops the notebook. At the end, download `sizecheck_llama31_8b.zip` and hand it back.
Takes about 30-40 minutes (16 GB download, then ~15-20 min of GPU work).

In [ ]:
# 1. GPU: needs at least ~40 GB; the result is only reported if the probe fits.
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU (A100)"
gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"{gib:.1f} GiB"); assert gib > 38, "this GPU is too small for two fp16 copies of an 8B model; use an A100"

In [ ]:
# 2. The pinned environment (the NRP image's torch 2.8.0 + torchvision 0.23.0, and the jobs' PINS). Restart is NOT needed.
# torchvision must match torch: Colab's own build crashes transformers' import (torchvision::nms does not exist).
!pip -q install "torch==2.8.0" "torchvision==0.23.0" "transformers==4.56.1" "accelerate==1.14.0" "datasets==4.3.0" safetensors numpy huggingface_hub
# the harness runs in fresh processes, which see the pinned versions (this kernel imported Colab's torch in step 1)
!python -c "import torch, transformers; print('pinned:', torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))"

In [ ]:
# 3. The harness at its pinned commit (sparse: benchmarks/weight_observer only).
import os, subprocess
C = "37ac5cb196913b2936e7a16d2d0c6f266f8af9c2"
if not os.path.exists("/content/tqp"):
    subprocess.run(["git", "clone", "-q", "--filter=blob:none", "--no-checkout",
                    "https://github.com/ahb-sjsu/turboquant-pro", "/content/tqp"], check=True)
    subprocess.run(["git", "-C", "/content/tqp", "sparse-checkout", "set", "--no-cone", "benchmarks/weight_observer"], check=True)
    subprocess.run(["git", "-C", "/content/tqp", "checkout", "-q", C], check=True)
head = subprocess.run(["git", "-C", "/content/tqp", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head == C, head
os.environ["PYTHONPATH"] = "/content/tqp/benchmarks"; os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"; os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("harness", head)

In [ ]:
# 4. The text, restaged and checked byte-for-byte against the prereg's sha256.
import hashlib
!cd /content/tqp/benchmarks && python -m weight_observer.stage_text --dest /content/wo/text
PIN = {"train.txt": "aee724fa58bfbdeb3fc6803297fb6bab27b203d7c40b39ddef9b9770e5d52fe5",
       "test.txt": "696cca6b65a171b0a358a4be6732cdfdf2dd6164a32e20fd70e3c13fc4dfae83"}
for n, h in PIN.items():
    got = hashlib.sha256(open(f"/content/wo/text/{n}", "rb").read()).hexdigest()
    assert got == h, (n, got); print(n, "matches the prereg")

In [ ]:
# 5. The model at the prereg's mirror commit (16.1 GB), with the staging manifest checked.
import json
!cd /content/tqp/benchmarks && python -m weight_observer.stage_models --model-id unsloth/Meta-Llama-3.1-8B --revision e9a141a2091ea561b96483212645a2a05e6f99fc --dest /content/wo/models/llama3.1-8b
st = json.load(open("/content/wo/models/llama3.1-8b/STAGED.json"))
assert st["revision"] == "e9a141a2091ea561b96483212645a2a05e6f99fc", st["revision"]; print("staged", st["revision"], round(st["bytes"] / 1e9, 2), "GB")

In [ ]:
# 6. G0 on this GPU, then the shape probe and the reference check (the NRP job's exact commands); stops at the first failure.
import subprocess
OUT = "/content/wo/sizecheck/llama3.1-8b"
M = "/content/wo/models/llama3.1-8b"
for args in (["weight_observer.g0_device", "--model-path", M, "--out", OUT],
             ["weight_observer.sizecheck", "memprobe", "--model-path", M, "--out", OUT],
             ["weight_observer.sizecheck", "refcheck", "--model-path", M, "--text", "/content/wo/text", "--out", OUT],
             # fp32 reference (32 GB) + fp16 harness (16 GB) fit an 80 GB A100: the fixed RULE, as for Gemma-2
             ["weight_observer.sizecheck", "refcheck", "--model-path", M, "--text", "/content/wo/text", "--out", OUT,
              "--reference", "float32"]):
    print(">>", " ".join(args[:2]), flush=True)
    subprocess.run(["python", "-m", *args], cwd="/content/tqp/benchmarks", check=True)

In [ ]:
# 7. Summary, checks against the prereg, and the bundle to hand back.
import json, platform, shutil
m = json.load(open(f"{OUT}/memprobe.json")); r = json.load(open(f"{OUT}/refcheck.json"))
assert r["evaluation_windows"]["sha256"] == "2eda697bba5e21a5f25b95c50528b3325025ffafc2c7ef020a1e2211af163223", "scored windows differ from the prereg"
peak = max(m["tables"]["reserved_gib"], *(a["reserved_gib"] for a in m["arms"].values())) + m["tables"]["context_gib"]
print("GPU", m["environment"]["gpu"], "| total", m["total_gib"], "GiB")
print("tables", m["tables"]["reserved_gib"], "| arms", {k: v["reserved_gib"] for k, v in m["arms"].items()}, "| context", m["tables"]["context_gib"])
print(f"peak incl. context {peak:.2f} GiB -> fits a 48 GB card (~44.9 GiB): {peak < 44.9}")
print("refcheck (bf16 reference, informational):", {k: r["verdict"][k] for k in ("kl_harness_per_token", "fp16_range_used", "nonfinite")})
r32 = json.load(open(f"{OUT}/refcheck_float32.json"))
print("refcheck against fp32 (the fixed RULE):", r32["verdict"]["kl_harness_per_token"], r32["verdict"]["checks"], "keep:", r32["verdict"]["keep_harness"])
json.dump({"colab": True, "python": platform.python_version()}, open(f"{OUT}/colab.json", "w"))
shutil.make_archive("/content/sizecheck_llama31_8b", "zip", OUT)
from google.colab import files; files.download("/content/sizecheck_llama31_8b.zip")